In [ ]:
"""Woorden per bericht, per leeftijdsgroep.

Sluit aan op goad_toolkit.visualizer: de klasse hieronder is een gewone
BasePlot-subclass, gebruikt `PlotSettings.highlight` / `base_color` /
`highlight_color` voor de kleurkeuze, en legt de gemiddelde-labels erop met de
bestaande `Annotate`-laag via `plot_on`.
"""

from pathlib import Path
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager

from goad_toolkit.visualizer import Annotate, BasePlot, PlotSettings

# --- data --------------------------------------------------------------------
# Kolommen: author (geanonimiseerd), leeftijd, n_words_av.
RELATIEF = Path("data") / "visuals_data" / "samengevoegd_leeftijd.csv"


def _startpunt() -> Path:
    """Map om vanaf te zoeken: het scriptbestand, of de werkmap in een notebook.

    In een notebook bestaat __file__ niet, vandaar de NameError-afhandeling.
    """
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd().resolve()


def zoek_csv(relatief: Path = RELATIEF) -> Path:
    """Loop van de start-map omhoog tot de repo-root met data/visuals_data/."""
    start = _startpunt()
    for map_ in [start, *start.parents]:
        kandidaat = map_ / relatief
        if kandidaat.exists():
            return kandidaat
    raise FileNotFoundError(
        f"'{relatief}' niet gevonden vanaf {start} of hoger. Geef het pad mee: "
        "lees_data(Path('...'))"
    )


def lees_data(pad: Optional[Path] = None) -> pd.DataFrame:
    """Lees de geaggregeerde data in: één rij per persoon per leeftijdsjaar."""
    pad = Path(pad) if pad is not None else zoek_csv()
    if not pad.exists():
        raise FileNotFoundError(
            f"Data niet gevonden op {pad}. Zorg dat de CSV meegecommit is "
            "(git check-ignore -v <pad>) of geef een ander pad mee."
        )
    data = pd.read_csv(pad)
    nodig = {"author", "leeftijd", "n_words_av"}
    ontbreekt = nodig - set(data.columns)
    if ontbreekt:
        raise ValueError(f"Kolom(men) {sorted(ontbreekt)} niet in {pad.name}")
    return data

# --- thema (los van de plotlogica, zodat het over elke BasePlot heen kan) -----
GROND = "#101827"
INK = "#FFFFFF"
INK_ZACHT = "#A6B0BF"
INK_DOF = "#8B96AA"
GRID = "#1B2433"
AS = "#39455A"

_beschikbaar = {f.name for f in font_manager.fontManager.ttflist}
SANS = "Instrument Sans" if "Instrument Sans" in _beschikbaar else "DejaVu Sans"
SERIF = "Instrument Serif" if "Instrument Serif" in _beschikbaar else "DejaVu Serif"


def donker_thema(fig, ax, grid_axis: str = "x") -> None:
    """Donkere grond, gridlijnen op één as, geen kader."""
    fig.patch.set_facecolor(GROND)
    ax.set_facecolor(GROND)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, color=GRID, linewidth=1, alpha=1)
    for kant in ("top", "right", "left"):
        ax.spines[kant].set_visible(False)
    ax.spines["bottom"].set_color(AS)
    ax.tick_params(colors=INK_DOF, labelsize=10, length=0)


def leeftijdsgroepen(
    data: pd.DataFrame, leeftijd: str = "leeftijd", gat: int = 5
) -> pd.Series:
    """Deel de leeftijden in groepen op, gesplitst waar een gat > `gat` jaar valt.

    De steekproef is niet continu over de leeftijd verdeeld; deze functie leest
    de clusters uit de data in plaats van ze hard te coderen.
    """
    uniek = np.sort(data[leeftijd].unique())
    grenzen: List[float] = [uniek[0]]
    for vorige, huidige in zip(uniek[:-1], uniek[1:]):
        if huidige - vorige > gat:
            grenzen.append(vorige)
            grenzen.append(huidige)
    grenzen.append(uniek[-1])

    paren = [(grenzen[i], grenzen[i + 1]) for i in range(0, len(grenzen), 2)]
    labels = [f"{int(lo)}–{int(hi)} jaar" for lo, hi in paren]

    def label_van(waarde):
        for (lo, hi), label in zip(paren, labels):
            if lo <= waarde <= hi:
                return label
        return labels[-1]

    return pd.Categorical(
        data[leeftijd].map(label_van), categories=labels, ordered=True
    )


class WoordenPerLeeftijdsgroep(BasePlot):
    """Eén punt per meting, per leeftijdsgroep, met het groepsgemiddelde erbij.

    Geen doorlopende leeftijdsas: de steekproef bestaat uit losse clusters, en
    een continue as zou een verband suggereren dat niet gemeten is. Kleur wijst
    aan welke groep de bewering draagt (`settings.highlight`), de rest is grijs.
    """

    def build(
        self,
        data: pd.DataFrame,
        x: str,
        group: str,
        person: Optional[str] = None,
        order: Optional[List[str]] = None,
        jitter: float = 0.22,
        markersize: float = 70,
        show_means: bool = True,
        **kwargs,
    ):
        """
        Parameters
        ----------
        data : pd.DataFrame
            Frame met één rij per meting.
        x : str
            Numerieke kolom (hier: gemiddeld aantal woorden per bericht).
        group : str
            Kolom met de leeftijdsgroep; wordt de y-as.
        person : Optional[str]
            Kolom met de persoon. Alleen gebruikt om n in de aslabels te zetten.
        order : Optional[List[str]]
            Volgorde van de groepen. Categorie-volgorde of sortering als None.
        jitter : float
            Verticale spreiding van de punten binnen een rij.
        markersize : float
            Puntgrootte (`s` in `ax.scatter`).
        show_means : bool
            Groepsgemiddelde als streep met het getal erboven.
        **kwargs : doorgegeven aan `ax.scatter`.
        """
        if self.ax is None:
            raise ValueError("No axes available for plotting")
        ontbreekt = [k for k in (x, group) if k not in data.columns]
        if ontbreekt:
            raise ValueError(f"Column(s) {ontbreekt} not in the data")

        if order is not None:
            groepen = list(order)
        elif isinstance(data[group].dtype, pd.CategoricalDtype):
            groepen = [g for g in data[group].cat.categories if g in set(data[group])]
        else:
            groepen = sorted(data[group].unique())

        highlight = set(self.settings.highlight)

        for i, groep in enumerate(groepen):
            deel = data[data[group] == groep]
            kleur = (
                self.settings.highlight_color
                if groep in highlight
                else self.settings.base_color
            )
            # vaste, cyclische spreiding: geen random klontering, en de plot
            # ziet er bij elke run hetzelfde uit
            stappen = np.array([0, 1, -1, 2, -2]) / 2 * jitter
            offsets = stappen[np.arange(len(deel)) % len(stappen)]
            self.ax.scatter(
                deel[x],
                i + offsets,
                s=markersize,
                color=kleur,
                alpha=0.55,
                linewidths=0,
                zorder=2,
                **kwargs,
            )

            if show_means:
                gemiddelde = deel[x].mean()
                streepkleur = kleur if groep in highlight else INK
                self.ax.vlines(
                    gemiddelde, i - 0.30, i + 0.30,
                    color=streepkleur, linewidth=3, zorder=3,
                )
                # de bevinding opgeschreven, met de bestaande Annotate-laag
                self.plot_on(
                    Annotate(self.settings),
                    text=f"{gemiddelde:.1f}".replace(".", ",") + " woorden",
                    xy=(gemiddelde, i - 0.38),
                    ha="center",
                    va="bottom",
                    fontsize=10.5,
                    fontweight="medium",
                    color=streepkleur,
                )

        # y-as: groepsnaam en aantallen als twee losse regels, zodat ze
        # verschillende grootte en kleur kunnen hebben
        self.ax.set_yticks([])
        blend = self.ax.get_yaxis_transform()
        for i, groep in enumerate(groepen):
            deel = data[data[group] == groep]
            naamkleur = (
                self.settings.highlight_color if groep in highlight else INK
            )
            self.ax.text(
                -0.015, i - 0.08, str(groep), transform=blend,
                ha="right", va="bottom", fontsize=13, color=naamkleur, clip_on=False,
            )
            if person is not None:
                n_personen = deel[person].nunique()
                pers = "persoon" if n_personen == 1 else "personen"
                jaar = "leeftijdsjaar" if len(deel) == 1 else "leeftijdsjaren"
                regel = f"{n_personen} {pers}, {len(deel)} {jaar}"
            else:
                jaar = "leeftijdsjaar" if len(deel) == 1 else "leeftijdsjaren"
                regel = f"{len(deel)} {jaar}"
            self.ax.text(
                -0.015, i + 0.10, regel, transform=blend,
                ha="right", va="top", fontsize=10, color=INK_DOF, clip_on=False,
            )

        self.ax.set_ylim(len(groepen) - 0.45, -0.75)  # jongste groep bovenaan
        # ronde bovengrens met ticks per 5, zoals in de HTML-versie
        boven = int(np.ceil(data[x].max() / 5.0) * 5)
        self.ax.set_xlim(0, boven)
        self.ax.set_xticks(np.arange(0, boven + 1, 5))

        donker_thema(self.fig, self.ax, grid_axis="x")

        self.ax.set_title("")
        self.fig.text(
            0.06, 0.955, self.settings.title,
            color=INK, fontfamily=SERIF, fontsize=17, ha="left", va="top",
        )
        self.fig.text(
            0.06, 0.890, SUBTITEL,
            color=INK_ZACHT, fontsize=9.5, ha="left", va="top", linespacing=1.5,
        )
        self.fig.text(
            0.06, 0.035, VOETNOOT,
            color="#8B95A5", fontsize=8, ha="left", va="bottom", linespacing=1.6,
        )
        if self.settings.xlabel is not None:
            self.ax.set_xlabel(
                self.settings.xlabel.upper(), color=INK_ZACHT, fontsize=9, labelpad=10
            )
        self.ax.set_ylabel("")

        self.fig.subplots_adjust(top=0.75, bottom=0.22, left=0.23, right=0.97)
        return self.fig, self.ax


SUBTITEL = (
    "Elk punt is het gemiddelde van één persoon in één leeftijdsjaar; de streep is het\n"
    "groepsgemiddelde. De steekproef valt uiteen in drie leeftijdsgroepen, dus er is geen\n"
    "doorlopende leeftijdsas."
)
VOETNOOT = (
    "Gebaseerd op 13.608 berichten van 8 personen. Het verschil tussen kinderen en volwassenen is\n"
    "groter dan de spreiding binnen die groepen. De twee oudere groepen overlappen vrijwel volledig,\n"
    "en de oudste bestaat uit één persoon."
)

plt.rcParams.update({"font.family": SANS, "savefig.facecolor": GROND})

samengevoegd = lees_data().assign(
    leeftijdsgroep=lambda frame: leeftijdsgroepen(frame, "leeftijd")
)
jongste = samengevoegd["leeftijdsgroep"].cat.categories[0]

settings = PlotSettings(
    figsize=(11, 6),
    title="Kinderen schrijven kortere berichten dan volwassenen",
    xlabel="gemiddeld aantal woorden per bericht",
    highlight=[jongste],
    highlight_color="#FF8A73",
    base_color="#8B96AA",
)

fig, ax = WoordenPerLeeftijdsgroep(settings).plot(
    data=samengevoegd,
    x="n_words_av",
    group="leeftijdsgroep",
    person="author",
)

fig.savefig(
    _startpunt() / "woorden_per_leeftijdsgroep.png",
    dpi=200, facecolor=GROND,
)
plt.show()
